# ARMS 消融研究与学术实证交互式笔记本
### 《Complexity Trap》去偏滚动行走 (Purged WFO) 及 5 阶段级联风控策略复现
本笔记本对应论文的 **Section 5 (Empirical Results)**。您可以通过逐步运行单元格，在本地复现消融研究指标、分年度业绩及图表。

## 1. 策略初始化与数据加载

In [1]:
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
%matplotlib inline

sys.path.append(os.getcwd())
import strategy
from run_full_academic_pipeline import custom_ablation_backtest, compute_buy_and_hold
print("✓ 基础依赖库加载成功！")

config = strategy.StrategyConfig()
config.portfolio_capital = 1_000_000.0
config.n_shuffles = 20
CACHE_FILE = "sci_baostock_assets_2016_2023.pkl"

print(f"[CACHE] 正在读取本地数据序列缓存 {CACHE_FILE} ...")
with open(CACHE_FILE, 'rb') as f:
    all_assets = pickle.load(f)
print(f"✓ 数据读取完成。共有 {len(all_assets)} 只 A 股个股的复权日线价格及 RF 模型预测信号。")

✓ 基础依赖库加载成功！
[CACHE] 正在读取本地数据序列缓存 sci_baostock_assets_2016_2023.pkl ...
✓ 数据读取完成。共有 104 只 A 股个股的复权日线价格及 RF 模型预测信号。


## 2. 运行被动基准与传统因子基准

In [2]:
print("--- 1. 被动买入持有基准 (Buy & Hold) ---")
bh_ret, bh_mdd, bh_calmar, bh_vals, bh_dates = compute_buy_and_hold(all_assets, config.portfolio_capital)
print(f"   Buy & Hold Return: {bh_ret*100:.2f}%, MDD: {bh_mdd*100:.2f}%, Calmar: {bh_calmar:.4f}")

print("\n--- 2. 传统趋势/动量因子基准 ---")
ma_ret, ma_mdd, ma_sharpe, ma_calmar, ma_vals, ma_dates, _ = custom_ablation_backtest(
    all_assets, config, strategy_type='ma_crossover', 
    enable_atr=False, enable_bbl=False, enable_toce=False, enable_trailing=False, enable_bbi_tp=False
)
print(f"   MA Crossover (5/20) Return: {ma_ret*100:.2f}%, MDD: {ma_mdd*100:.2f}%, Sharpe: {ma_sharpe:.4f}, Calmar: {ma_calmar:.4f}")

ma_atr_ret, ma_atr_mdd, ma_atr_sharpe, ma_atr_calmar, _, _, _ = custom_ablation_backtest(
    all_assets, config, strategy_type='ma_crossover', 
    enable_atr=True, enable_bbl=False, enable_toce=False, enable_trailing=False, enable_bbi_tp=False
)
print(f"   MA + ATR Crossover Return: {ma_atr_ret*100:.2f}%, MDD: {ma_atr_mdd*100:.2f}%, Sharpe: {ma_atr_sharpe:.4f}, Calmar: {ma_atr_calmar:.4f}")

mom_ret, mom_mdd, mom_sharpe, mom_calmar, _, _, _ = custom_ablation_backtest(
    all_assets, config, strategy_type='momentum', 
    enable_atr=False, enable_bbl=False, enable_toce=False, enable_trailing=False, enable_bbi_tp=False
)
print(f"   Momentum (12M-1M) Return: {mom_ret*100:.2f}%, MDD: {mom_mdd*100:.2f}%, Sharpe: {mom_sharpe:.4f}, Calmar: {mom_calmar:.4f}")

--- 1. 被动买入持有基准 (Buy & Hold) ---
   Buy & Hold Return: 44.33%, MDD: -23.35%, Calmar: 0.3928

--- 2. 传统趋势/动量因子基准 ---
   MA Crossover (5/20) Return: 92.09%, MDD: -45.60%, Sharpe: 0.2294, Calmar: 0.3704
   MA + ATR Crossover Return: 99.35%, MDD: -45.64%, Sharpe: 0.2454, Calmar: 0.3930
   Momentum (12M-1M) Return: 10.19%, MDD: -50.62%, Sharpe: 0.0122, Calmar: 0.0464


## 3. 运行消融对照组 (System 0-A 至 System 5)

In [3]:
results = []

print("   [0-A] Pure Rules (纯规则 S4)...", end="")
r0a_ret, r0a_mdd, r0a_sharpe, r0a_calmar, r0a_vals, _, _ = custom_ablation_backtest(all_assets, config, strategy_type='pure_rules', enable_atr=True, enable_bbl=True, enable_toce=True, enable_trailing=False, enable_bbi_tp=False)
results.append({'System': 'System 0-A: Pure Rules (S4 Equivalent)', 'Return': r0a_ret, 'MDD': r0a_mdd, 'Sharpe': r0a_sharpe, 'Calmar': r0a_calmar})
print(" 完成。")

print("   [0-B] Pure Rules (纯规则 S5)...", end="")
r0b_ret, r0b_mdd, r0b_sharpe, r0b_calmar, _, _, _ = custom_ablation_backtest(all_assets, config, strategy_type='pure_rules', enable_atr=True, enable_bbl=True, enable_toce=True, enable_trailing=True, enable_bbi_tp=True)
results.append({'System': 'System 0-B: Pure Rules (S5 Equivalent)', 'Return': r0b_ret, 'MDD': r0b_mdd, 'Sharpe': r0b_sharpe, 'Calmar': r0b_calmar})
print(" 完成。")

print("   [1] ML Only (纯ML基准)...", end="")
r1_ret, r1_mdd, r1_sharpe, r1_calmar, r1_vals, _, _ = custom_ablation_backtest(all_assets, config, strategy_type='ml_rules', enable_atr=False, enable_bbl=False, enable_toce=False, enable_trailing=False, enable_bbi_tp=False)
results.append({'System': 'System 1: Pure ML Baseline', 'Return': r1_ret, 'MDD': r1_mdd, 'Sharpe': r1_sharpe, 'Calmar': r1_calmar})
print(" 完成。")

print("   [2] ML + ATR Stop (ML加动态止损)...", end="")
r2_ret, r2_mdd, r2_sharpe, r2_calmar, _, _, _ = custom_ablation_backtest(all_assets, config, strategy_type='ml_rules', enable_atr=True, enable_bbl=False, enable_toce=False, enable_trailing=False, enable_bbi_tp=False)
results.append({'System': 'System 2: ML + ATR Stop', 'Return': r2_ret, 'MDD': r2_mdd, 'Sharpe': r2_sharpe, 'Calmar': r2_calmar})
print(" 完成。")

print("   [3] ML + ATR + BBL...", end="")
r3_ret, r3_mdd, r3_sharpe, r3_calmar, _, _, _ = custom_ablation_backtest(all_assets, config, strategy_type='ml_rules', enable_atr=True, enable_bbl=True, enable_toce=False, enable_trailing=False, enable_bbi_tp=False)
results.append({'System': 'System 3: ML + ATR + BBL', 'Return': r3_ret, 'MDD': r3_mdd, 'Sharpe': r3_sharpe, 'Calmar': r3_calmar})
print(" 完成。")

print("   [4] ML + ATR + BBL + TOCE (核心机制)...", end="")
r4_ret, r4_mdd, r4_sharpe, r4_calmar, r4_vals, _, _ = custom_ablation_backtest(all_assets, config, strategy_type='ml_rules', enable_atr=True, enable_bbl=True, enable_toce=True, enable_trailing=False, enable_bbi_tp=False)
results.append({'System': 'System 4: ML + ATR + BBL + TOCE', 'Return': r4_ret, 'MDD': r4_mdd, 'Sharpe': r4_sharpe, 'Calmar': r4_calmar})
print(" 完成。")

print("   [5] Full ARMS Framework (完整框架)...", end="")
r5_ret, r5_mdd, r5_sharpe, r5_calmar, s5_vals, s5_dates, s5_trade_log = custom_ablation_backtest(all_assets, config, strategy_type='ml_rules', enable_atr=True, enable_bbl=True, enable_toce=True, enable_trailing=True, enable_bbi_tp=True)
results.append({'System': 'System 5: Full ARMS Framework', 'Return': r5_ret, 'MDD': r5_mdd, 'Sharpe': r5_sharpe, 'Calmar': r5_calmar})
print(" 完成。")

   [0-A] Pure Rules (纯规则 S4)... 完成。
   [0-B] Pure Rules (纯规则 S5)... 完成。
   [1] ML Only (纯ML基准)... 完成。
   [2] ML + ATR Stop (ML加动态止损)... 完成。
   [3] ML + ATR + BBL... 完成。
   [4] ML + ATR + BBL + TOCE (核心机制)... 完成。
   [5] Full ARMS Framework (完整框架)... 完成。


## 4. 整理消融实验数据表 (对标表 3)

In [4]:
df_res = pd.DataFrame(results)
df_res['Return'] = df_res['Return'].apply(lambda x: f"{x*100:.2f}%")
df_res['MDD'] = df_res['MDD'].apply(lambda x: f"{x*100:.2f}%")
df_res['Sharpe'] = df_res['Sharpe'].apply(lambda x: f"{x:.4f}" if pd.notnull(x) else "N/A")
df_res['Calmar'] = df_res['Calmar'].apply(lambda x: f"{x:.4f}")
df_res

,System,Return,MDD,Sharpe,Calmar
0,System 0-A: Pure Rules (S4 Equivalent),-66.71%,-79.77%,-0.6653,-0.2899
1,System 0-B: Pure Rules (S5 Equivalent),-43.33%,-62.30%,-0.5955,-0.2038
2,System 1: Pure ML Baseline,-0.54%,-32.04%,-0.1593,-0.0040
3,System 2: ML + ATR Stop,-23.60%,-32.22%,-0.5010,-0.1934
4,System 3: ML + ATR + BBL,-12.90%,-43.06%,-0.3205,-0.0754
5,System 4: ML + ATR + BBL + TOCE,-3.89%,-29.67%,-0.1960,-0.0318
6,System 5: Full ARMS Framework,3.55%,-16.68%,-0.1129,0.0502


## 5. 绘制真实净值曲线 (对标图 2)

In [5]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
dates_dt = pd.to_datetime(s5_dates)

ax.plot(dates_dt, np.array(r0a_vals) / r0a_vals[0], label='System 0-A (纯规则 S4)', color='#8B0000', linestyle='--')
ax.plot(dates_dt, np.array(r1_vals) / r1_vals[0], label='System 1 (纯ML)', color='#FF6B35')
ax.plot(dates_dt, np.array(r4_vals) / r4_vals[0], label='System 4 (ML+ATR+BBL+TOCE)', color='#FFA500')
ax.plot(dates_dt, np.array(s5_vals) / s5_vals[0], label='System 5 (完整ARMS)', color='#2196F3', linewidth=2.5)
ax.plot(dates_dt, np.array(ma_vals) / ma_vals[0], label='MA均线交叉', color='#4CAF50', linewidth=2.0)
ax.plot(dates_dt, np.array(bh_vals) / bh_vals[0], label='B&H', color='#9E9E9E', linestyle=':')

ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('年份', fontsize=12)
ax.set_ylabel('账户净值', fontsize=12)
ax.set_title('六阶段消融研究累计净值曲线对比 (2019-2023，117只随机个股真实数据)', fontsize=14, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
plt.show()